<a href="https://colab.research.google.com/github/SarikMaj/shapely-project/blob/main/shapely_projekt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Knihovna Shapely – geometrie v Pythonu

**Semestrální projekt – představení knihovny**

Obsah prezentace (cca 20 minut):

1. Co je Shapely a kam patří v ekosystému prostorových knihoven
2. Geometrické objekty a jejich vlastnosti
3. Prostorové vztahy (predikáty)
4. Geometrické operace
5. **Praktická ukázka na reálných datech** – Česká republika, její města a řeky
6. **Limity knihovny** – na co si dát pozor
7. Alternativy, shrnutí a zdroje

## 1. Co je Shapely?

**Shapely** je knihovna pro práci s **planární (rovinnou) geometrií** – body, liniemi a polygony. Umí:

* vytvářet geometrie a zjišťovat jejich vlastnosti (plocha, délka, obalový obdélník…),
* testovat **prostorové vztahy** (protíná se? leží uvnitř? dotýká se?),
* provádět **geometrické operace** (buffer, průnik, sjednocení, rozdíl, zjednodušení…).

**Jak funguje uvnitř?** Shapely je „pythonní obal“ nad C++ knihovnou **GEOS** (Geometry Engine – Open Source), což je port javové knihovny **JTS**. Stejný GEOS používá i QGIS, PostGIS nebo GDAL – výsledky jsou proto konzistentní napříč GIS nástroji.

**Kde se s ní potkáte?** Prakticky všude: každý prvek ve sloupci `geometry` v **GeoPandas** je objekt Shapely. Kdo používá GeoPandas, používá (často nevědomky) Shapely.

**Co Shapely záměrně *neumí*:**

| Neumí | Kdo to řeší |
|---|---|
| číst/zapisovat soubory (SHP, GPKG, GeoJSON) | GeoPandas, Fiona, pyogrio |
| souřadnicové systémy (CRS) a jejich převody | pyproj |
| atributové tabulky | GeoPandas |
| rastrová data | rasterio |
| vykreslování map | matplotlib, GeoPandas |

> **Shapely 2.0** (2022) přinesla velkou změnu: knihovna PyGEOS se sloučila se Shapely a přibyly **vektorizované funkce** pracující nad celými NumPy poli geometrií. Ty jsou výrazně rychlejší než cykly v Pythonu – ukážeme si to dále.

## 2. Instalace a import

In [ ]:
# V Google Colab je Shapely i GeoPandas předinstalovaná, lokálně stačí:
# !pip install shapely geopandas pyproj matplotlib

import time
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import pyproj
import shapely
from shapely import Point, LineString, Polygon, MultiPoint, MultiLineString, MultiPolygon
from shapely.plotting import plot_polygon, plot_line, plot_points

import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning, module="pyogrio")   # nevýznamné varování při čtení GeoJSON

print("Shapely:", shapely.__version__)
print("GEOS:   ", shapely.geos_version_string)
print("GeoPandas:", gpd.__version__)

## 3. Geometrické objekty

Shapely implementuje model **Simple Features** (standard OGC). Základní typy:

| Typ | Popis | Příklad v praxi |
|---|---|---|
| `Point` | bod | město, měřicí stanice |
| `LineString` | lomená čára | řeka, silnice |
| `LinearRing` | uzavřená čára | hranice polygonu |
| `Polygon` | plocha (může mít díry) | stát, jezero, budova |
| `MultiPoint`, `MultiLineString`, `MultiPolygon` | kolekce stejného typu | souostroví |
| `GeometryCollection` | kolekce různých typů | výsledek některých operací |

In [ ]:
bod = Point(2, 3)
cara = LineString([(0, 0), (2, 1), (4, 0), (6, 2)])

# polygon s dírou: první seznam = vnější obvod, druhý = seznam děr
polygon = Polygon(
    shell=[(0, 0), (6, 0), (6, 4), (0, 4)],
    holes=[[(2, 1), (4, 1), (4, 3), (2, 3)]],
)

print("Bod:     ", bod)
print("Čára:    ", cara)
print("Polygon: ", polygon)

Výpis výše je ve formátu **WKT** (Well-Known Text) – textový zápis geometrie, kterému rozumí i PostGIS nebo QGIS. Geometrii lze z WKT také vytvořit a každá geometrie nabízí řadu atributů:

In [ ]:
g = shapely.from_wkt("POLYGON ((0 0, 6 0, 6 4, 0 4, 0 0), (2 1, 4 1, 4 3, 2 3, 2 1))")

print("Typ:              ", g.geom_type)
print("Plocha:           ", g.area)          # 24 - 4 (díra) = 20
print("Obvod:            ", g.length)        # vnější + vnitřní obvod
print("Obalový obdélník: ", g.bounds)        # (minx, miny, maxx, maxy)
print("Těžiště:          ", g.centroid)
print("Je validní?       ", g.is_valid)
print("Délka čáry:       ", round(cara.length, 3))
print("Souřadnice čáry:  ", list(cara.coords))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
plot_polygon(polygon, ax=ax, add_points=False, color="tab:green", alpha=0.4)
plot_line(cara, ax=ax, color="tab:blue", linewidth=2)
plot_points(bod, ax=ax, color="tab:red", markersize=10)
ax.set_title("Point, LineString a Polygon s dírou")
ax.set_aspect("equal")
plt.show()

## 4. Prostorové vztahy (predikáty)

Predikáty vracejí `True`/`False` a odpovídají na otázky typu *„leží město v ČR?“* nebo *„kříží řeka hranici?“*. Vycházejí z matematického modelu **DE-9IM**, který porovnává vnitřek, hranici a vnějšek dvou geometrií.

| Metoda | Otázka |
|---|---|
| `a.intersects(b)` | mají a a b cokoli společného? |
| `a.contains(b)` / `b.within(a)` | leží b celé uvnitř a? |
| `a.touches(b)` | dotýkají se jen hranicí? |
| `a.crosses(b)` | kříží se (např. čára přes polygon)? |
| `a.disjoint(b)` | nemají nic společného? |
| `a.relate(b)` | celá matice DE-9IM jako řetězec |

In [ ]:
ctverec = Polygon([(0, 0), (4, 0), (4, 4), (0, 4)])
soused = Polygon([(4, 0), (8, 0), (8, 4), (4, 4)])     # sdílí se čtvercem jednu stranu
uvnitr = Point(1, 1)
hranice_bod = Point(4, 2)
pres = LineString([(-1, 2), (5, 2)])

print("čtverec obsahuje bod (1,1):      ", ctverec.contains(uvnitr))
print("čtverec obsahuje bod na hranici: ", ctverec.contains(hranice_bod))   # pozor: False!
print("bod na hranici protíná čtverec:  ", ctverec.intersects(hranice_bod))
print("čtverec se dotýká souseda:       ", ctverec.touches(soused))
print("čára kříží čtverec:              ", pres.crosses(ctverec))
print("DE-9IM čtverec vs. soused:       ", ctverec.relate(soused))

> 💡 **Častá past:** bod ležící přesně *na hranici* polygonu není podle definice „uvnitř“ (`contains` vrací `False`), ale `intersects` vrací `True`. Pokud chcete zahrnout i hranici, použijte `covers` / `covered_by`.

## 5. Geometrické operace

Operace vracejí **novou geometrii**. Nejpoužívanější:

In [ ]:
a = Point(0, 0).buffer(2)          # kruh = buffer kolem bodu
b = Point(2, 0).buffer(2)

operace = {
    "vstup: a, b": MultiPolygon([a, b]),
    "průnik  a ∩ b": a.intersection(b),
    "sjednocení  a ∪ b": a.union(b),
    "rozdíl  a − b": a.difference(b),
    "sym. rozdíl  a △ b": a.symmetric_difference(b),
    "konvexní obal (a ∪ b)": a.union(b).convex_hull,
}

fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for ax, (nazev, geom) in zip(axes.flat, operace.items()):
    for x in (a, b):
        plot_polygon(x, ax=ax, add_points=False, facecolor="none", edgecolor="grey", linestyle="--")
    plot_polygon(geom, ax=ax, add_points=False, color="tab:orange", alpha=0.6)
    ax.set_title(nazev)
    ax.set_aspect("equal")
    ax.axis("off")
plt.tight_layout()
plt.show()

### Těžiště vs. reprezentativní bod

`centroid` je geometrické těžiště – u nekonvexních tvarů může ležet **mimo** polygon (typicky u států tvaru „C“ nebo u souostroví). Pokud potřebujete bod pro popisek, který leží určitě uvnitř, použijte `representative_point()`.

In [ ]:
podkova = Polygon([(0, 0), (6, 0), (6, 6), (0, 6), (0, 4), (4, 4), (4, 2), (0, 2)])

fig, ax = plt.subplots(figsize=(4, 4))
plot_polygon(podkova, ax=ax, add_points=False, color="lightgrey")
plot_points(podkova.centroid, ax=ax, color="red", markersize=10, label="centroid")
plot_points(podkova.representative_point(), ax=ax, color="green", markersize=10, label="representative_point")
ax.legend(loc="upper left", fontsize=8)
ax.set_aspect("equal")
plt.show()

print("Leží centroid uvnitř?", podkova.contains(podkova.centroid))
print("Leží representative_point uvnitř?", podkova.contains(podkova.representative_point()))

## 6. Praktická ukázka: Česká republika, její města a řeky

### Data

Použijeme otevřená data **[Natural Earth](https://www.naturalearthdata.com/)** v měřítku 1 : 10 milionů (public domain):

* `ne_10m_admin_0_countries` – hranice států světa (polygony),
* `ne_10m_populated_places_simple` – významná sídla světa (body),
* `ne_10m_rivers_lake_centerlines` + `ne_10m_rivers_europe` – řeky (linie).

Data se při prvním spuštění automaticky stáhnou z oficiálního GitHub repozitáře Natural Earth do složky `data/` (při dalším spuštění se už jen načtou).

> Shapely sama soubory číst **neumí**, proto k načtení použijeme GeoPandas. Hned potom ale z tabulky vytáhneme „čisté“ geometrie Shapely a dál pracujeme jen s nimi.

In [ ]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
URL = "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/{}.geojson"

def nacti(nazev):
    """Stáhne vrstvu Natural Earth (pokud ještě není stažená) a načte ji jako GeoDataFrame."""
    soubor = DATA_DIR / f"{nazev}.geojson"
    if not soubor.exists():
        print(f"Stahuji {nazev} …")
        urlretrieve(URL.format(nazev), soubor)
    gdf = gpd.read_file(soubor)
    gdf.columns = gdf.columns.str.lower()   # sjednocení názvů sloupců
    return gdf

staty = nacti("ne_10m_admin_0_countries")
mesta_svet = nacti("ne_10m_populated_places_simple")
# řeky ze dvou vrstev (hlavní toky světa + podrobnější evropské) spojíme do jedné tabulky
reky_svet = pd.concat(
    [nacti("ne_10m_rivers_lake_centerlines"), nacti("ne_10m_rivers_europe")], ignore_index=True
)

print(f"Států: {len(staty)}, sídel: {len(mesta_svet)}, úseků řek: {len(reky_svet)}")
print("CRS:", staty.crs)

Z GeoDataFrame vytáhneme geometrii ČR – je to obyčejný objekt Shapely:

In [ ]:
cr = staty.loc[staty["adm0_a3"] == "CZE", "geometry"].iloc[0]

print("Typ objektu:      ", type(cr))
print("Typ geometrie:    ", cr.geom_type)
print("Počet vrcholů:    ", shapely.get_num_coordinates(cr))
print("Obalový obdélník: ", [round(v, 2) for v in cr.bounds])
cr

> Objekty Shapely se v Jupyteru umí samy vykreslit jako SVG – stačí je napsat na konec buňky.

### 6.1 Pozor na souřadnicový systém! (první limit)

Data jsou ve **WGS 84 (EPSG:4326)** – souřadnice jsou zeměpisné **stupně**. Shapely ale o souřadnicových systémech nic neví a vše počítá v **rovině**. Zkusme spočítat rozlohu ČR:

In [ ]:
print(f"Rozloha ČR podle Shapely ve WGS 84: {cr.area:.4f}  (jednotky: 'čtvereční stupně' = nesmysl)")

Abychom dostali smysluplné hodnoty v metrech, musíme geometrii **převést do projektovaného souřadnicového systému**. Pro ČR je přirozenou volbou **S-JTSK / Křovák East North (EPSG:5514)**. Převod souřadnic dělá knihovna **pyproj**, Shapely jen aplikuje funkci na všechny vrcholy pomocí `shapely.transform`:

In [ ]:
transformer = pyproj.Transformer.from_crs("EPSG:4326", "EPSG:5514", always_xy=True)

def do_krovaka(geom):
    """Převede geometrii (nebo pole geometrií) z WGS 84 do S-JTSK (EPSG:5514)."""
    return shapely.transform(geom, lambda xy: np.column_stack(transformer.transform(xy[:, 0], xy[:, 1])))

cr_sjtsk = do_krovaka(cr)

print(f"Rozloha ČR (S-JTSK): {cr_sjtsk.area / 1e6:,.0f} km²   (oficiálně 78 871 km²)")
print(f"Délka hranic:        {cr_sjtsk.length / 1e3:,.0f} km   (oficiálně cca 2 290 km)")

Výsledek je velmi blízko oficiální hodnotě – drobný rozdíl je způsoben **generalizací** dat (měřítko 1 : 10 mil.). U délky hranic je rozdíl větší: generalizovaná hranice je „hladší“, a tedy kratší (tzv. *paradox pobřeží*).

### 6.2 Která města leží v ČR? – vektorizované predikáty

Datová sada sídel je celosvětová (~7 300 bodů). Města v ČR najdeme prostorovým dotazem `contains`. Ukážeme si dvě cesty – **klasický cyklus** v Pythonu a **vektorizovanou funkci** Shapely 2.0:

In [ ]:
body = mesta_svet.geometry.values      # NumPy pole objektů Shapely

# a) klasický Python cyklus
t0 = time.perf_counter()
maska_cyklus = [cr.contains(p) for p in body]
t_cyklus = time.perf_counter() - t0

# b) vektorizovaně + "připravená" geometrie (prepare = předpočítaný prostorový index hran polygonu)
shapely.prepare(cr)
t0 = time.perf_counter()
maska_vektor = shapely.contains(cr, body)
t_vektor = time.perf_counter() - t0

print(f"Cyklus:        {t_cyklus * 1000:7.1f} ms")
print(f"Vektorizovaně: {t_vektor * 1000:7.1f} ms   → cca {t_cyklus / t_vektor:.0f}× rychleji")
print("Výsledky shodné:", np.array_equal(maska_cyklus, maska_vektor))

mesta = mesta_svet[maska_vektor][["name", "pop_max", "geometry"]].sort_values("pop_max", ascending=False)
mesta = mesta.reset_index(drop=True)
mesta

### 6.3 Jak daleko je každé město od státní hranice?

`distance` vrací nejkratší vzdálenost, `shapely.ops.nearest_points` zase dvojici nejbližších bodů. Vzdálenosti počítáme v S-JTSK, aby vyšly v metrech. Hranice státu je `boundary` polygonu (tedy linie).

In [ ]:
from shapely.ops import nearest_points

mesta_sjtsk = do_krovaka(mesta.geometry.values)
hranice_sjtsk = cr_sjtsk.boundary

mesta["od_hranice_km"] = (shapely.distance(mesta_sjtsk, hranice_sjtsk) / 1000).round(1)
mesta[["name", "pop_max", "od_hranice_km"]].sort_values("od_hranice_km")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
plot_polygon(cr_sjtsk, ax=ax, add_points=False, color="whitesmoke", edgecolor="black")

for nazev, mesto in zip(mesta["name"], mesta_sjtsk):
    na_hranici = nearest_points(mesto, hranice_sjtsk)[1]       # nejbližší bod na hranici
    plot_line(LineString([mesto, na_hranici]), ax=ax, color="tab:red", linewidth=1, add_points=False)
    plot_points(mesto, ax=ax, color="black", markersize=4)
    ax.annotate(nazev, (mesto.x, mesto.y), xytext=(4, 4), textcoords="offset points", fontsize=8)

ax.set_title("Nejkratší spojnice velkých měst se státní hranicí")
ax.set_aspect("equal")
ax.axis("off")
plt.show()

### 6.4 Řeky: ořez na území ČR a délky toků

Z celosvětových řek vybereme ty, které protínají ČR, ořízneme je hranicí (`intersection`) a sloučíme úseky stejného jména (`union_all`).

In [ ]:
reky_geom = reky_svet.geometry.values
v_cr = shapely.intersects(cr, reky_geom)                     # rychlý výběr (cr je "prepared")
orez = shapely.intersection(reky_geom[v_cr], cr)             # ořez hranicí státu

reky = (
    gpd.GeoDataFrame({"name": reky_svet["name"].values[v_cr]}, geometry=orez, crs="EPSG:4326")
    .dissolve(by="name")                                     # GeoPandas uvnitř volá shapely.union_all
    .reset_index()
)
reky = reky[~reky.geometry.is_empty]
reky["geometry"] = do_krovaka(reky.geometry.values)
reky["delka_v_cr_km"] = (shapely.length(reky.geometry.values) / 1000).round(0)
reky.sort_values("delka_v_cr_km", ascending=False)[["name", "delka_v_cr_km"]]

> **Kritický pohled na data:** Vltava vyšla kratší než skutečných ~430 km (generalizace), naopak Svitava (reálně ~100 km) nebo Úhlava jsou zjevně příliš dlouhé – v Natural Earth jsou pod jejich jménem uloženy i úseky jiných toků. Shapely počítá správně, ale výsledek je jen tak dobrý, jak dobrá jsou vstupní data. Pro skutečnou analýzu by se použila podrobná data (např. DIBAVOD). Mimochodem: Natural Earth má i překlep „Pizen“ místo Plzeň.

### 6.5 Buffer: která města leží do 5 km od velké řeky?

Kolem všech řek vytvoříme pásmo 5 km (`buffer`), pásma sloučíme do jednoho polygonu a otestujeme, která města do něj padnou.

In [ ]:
pasmo = shapely.union_all(shapely.buffer(reky.geometry.values, 5_000))   # 5 km v metrech
pasmo = pasmo.intersection(cr_sjtsk)

u_reky = shapely.contains(pasmo, mesta_sjtsk)
print(f"Pásmo 5 km kolem řek pokrývá {pasmo.area / cr_sjtsk.area:.1%} území ČR.")
print("Města v pásmu:   ", ", ".join(mesta["name"][u_reky]))
print("Města mimo pásmo:", ", ".join(mesta["name"][~u_reky]))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
plot_polygon(cr_sjtsk, ax=ax, add_points=False, color="whitesmoke", edgecolor="black")
plot_polygon(pasmo, ax=ax, add_points=False, color="lightblue", alpha=0.7)
for geom in reky.geometry:
    plot_line(geom, ax=ax, color="tab:blue", linewidth=1, add_points=False)
plot_points(mesta_sjtsk[u_reky], ax=ax, color="tab:green", markersize=6)
plot_points(mesta_sjtsk[~u_reky], ax=ax, color="tab:red", markersize=6)
ax.set_title("Pásmo 5 km kolem hlavních řek (zeleně města v pásmu, červeně mimo)")
ax.set_aspect("equal")
ax.axis("off")
plt.show()

### 6.6 Voroného polygony: „spádové oblasti“ měst

Voroného diagram rozdělí rovinu tak, že každý bod území připadne **nejbližšímu** městu. Shapely ho spočítá jedinou funkcí; výsledek ořízneme hranicí ČR.

In [ ]:
voronoi = shapely.voronoi_polygons(MultiPoint(list(mesta_sjtsk)), extend_to=cr_sjtsk)
oblasti = [shapely.intersection(cr_sjtsk, bunka) for bunka in voronoi.geoms]

# Buňky vracené GEOS nejsou ve stejném pořadí jako body → přiřadíme je přes predikát contains
mesta["spadova_oblast_km2"] = [
    round(next(o for o in oblasti if o.contains(m)).area / 1e6) for m in mesta_sjtsk
]

fig, ax = plt.subplots(figsize=(11, 6))
barvy = plt.cm.tab20(np.linspace(0, 1, len(oblasti)))
for oblast, barva in zip(oblasti, barvy):
    plot_polygon(oblast, ax=ax, add_points=False, color=barva, alpha=0.6, edgecolor="white")
plot_points(mesta_sjtsk, ax=ax, color="black", markersize=4)
for nazev, m in zip(mesta["name"], mesta_sjtsk):
    ax.annotate(nazev, (m.x, m.y), xytext=(4, 4), textcoords="offset points", fontsize=8)
ax.set_title("Voroného polygony: území nejblíže danému městu")
ax.set_aspect("equal")
ax.axis("off")
plt.show()

mesta[["name", "pop_max", "spadova_oblast_km2"]].sort_values("spadova_oblast_km2", ascending=False)

### 6.7 Prostorový index STRtree: celosvětové prostorové spojení

Úloha: ke **každému** ze ~7 300 sídel světa zjistit, ve kterém ze ~260 států leží. Naivně by to znamenalo cca 1,9 milionu testů `contains`. **STRtree** (R-strom) nejdřív rychle vyřadí státy, jejichž obalový obdélník bod vůbec nezasahuje, a přesný test udělá jen pro pár kandidátů.

Tohle je přesně to, co dělá `geopandas.sjoin` – uvnitř volá STRtree ze Shapely.

In [ ]:
staty_geom = staty.geometry.values
strom = shapely.STRtree(staty_geom)

t0 = time.perf_counter()
idx_bod, idx_stat = strom.query(body, predicate="within")   # vrací páry indexů (bod, stát)
t_strom = time.perf_counter() - t0

# pro srovnání: naivní dvojitý cyklus na vzorku 200 bodů a odhad na celou sadu
t0 = time.perf_counter()
for p in body[:200]:
    for s in staty_geom:
        s.contains(p)
t_naivne = (time.perf_counter() - t0) / 200 * len(body)

print(f"STRtree:                   {t_strom:6.2f} s")  # dotaz + přesný test u kandidátů
print(f"Naivní cyklus (odhad):     {t_naivne:6.1f} s   → cca {t_naivne / t_strom:.0f}× pomaleji")
print(f"Nalezeno párů bod–stát:    {len(idx_bod)} z {len(body)} bodů")

Zrychlení není „nekonečné“, protože většinu času zabírá přesný test `within` nad podrobnými polygony států (Kanada, Rusko, Indonésie mají desítky tisíc vrcholů). Čím víc prvků a čím jednodušší geometrie, tím víc se strom vyplatí – u milionů bodů je rozdíl mezi sekundami a hodinami.

Zajímavé je, že **ne všechna sídla** se do nějakého státu „trefila“. Podívejme se, která to jsou:

In [ ]:
bez_statu = np.setdiff1d(np.arange(len(body)), idx_bod)
print(f"Sídel mimo všechny polygony: {len(bez_statu)}")
mesta_svet.iloc[bez_statu][["name", "adm0name"]].head(10)

Jde o přímořská a ostrovní sídla, jejichž bod leží kousek **za generalizovanou pobřežní linií**. Speciálním případem je stanice na jižním pólu: bod má zeměpisnou šířku přesně −90° a v „rovinném“ světě Shapely leží na samém okraji polygonu Antarktidy, takže test `within` neprojde – další ukázka toho, že Shapely nezná kulatou Zemi. Není to chyba Shapely, ale dat – přesto je to důležitá praktická zkušenost: při spojování dat různé přesnosti je často nutné použít toleranci (např. `dwithin` nebo `query_nearest` s `max_distance`).

### 6.8 Zjednodušení geometrie (Douglas–Peucker)

Pro webové mapy nebo přehledky se hodí geometrie zjednodušit. `simplify(tolerance)` odstraní vrcholy, které od výsledné linie neodbočují víc než o toleranci. `preserve_topology=True` zaručí, že výsledek zůstane validní.

In [ ]:
tolerance = [0, 1_000, 5_000, 20_000]     # metry
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, tol in zip(axes, tolerance):
    zj = cr_sjtsk.simplify(tol, preserve_topology=True) if tol else cr_sjtsk
    plot_polygon(zj, ax=ax, add_points=False, color="tab:purple", alpha=0.5)
    ax.set_title(f"tol = {tol / 1000:.0f} km\n{shapely.get_num_coordinates(zj)} vrcholů, "
                 f"{zj.area / 1e6:,.0f} km²", fontsize=9)
    ax.set_aspect("equal")
    ax.axis("off")
plt.show()

## 7. Limity knihovny Shapely

Některé limity jsme už viděli (CRS, generalizace dat). Projdeme je systematicky.

### 7.1 Žádný souřadnicový systém a jen rovinná geometrie

Shapely počítá **euklidovsky v rovině**. Geometrie nenese informaci o CRS a knihovna nepozná, že kombinujete stupně s metry. Vzdálenost Praha–Brno ve stupních nemá smysl, a ani převod „stupeň ≈ 111 km“ nefunguje, protože délka stupně zeměpisné délky závisí na zeměpisné šířce:

In [ ]:
praha = mesta.loc[mesta["name"] == "Prague", "geometry"].iloc[0]
brno = mesta.loc[mesta["name"] == "Brno", "geometry"].iloc[0]

d_stupne = praha.distance(brno)
d_sjtsk = do_krovaka(praha).distance(do_krovaka(brno))
d_geod = pyproj.Geod(ellps="WGS84").inv(praha.x, praha.y, brno.x, brno.y)[2]   # vzdálenost na elipsoidu

print(f"Shapely ve WGS 84:          {d_stupne:.3f} °  → × 111 km = {d_stupne * 111:.0f} km (špatně)")
print(f"Shapely v S-JTSK:           {d_sjtsk / 1000:.1f} km")
print(f"pyproj.Geod (na elipsoidu): {d_geod / 1000:.1f} km")

Pro malé území v dobré projekci je rozdíl zanedbatelný. Pro **celosvětové výpočty** (vzdálenosti mezi kontinenty, plochy velkých států, geometrie přes 180. poledník) ale Shapely správný výsledek nedá – tam je potřeba `pyproj.Geod` nebo knihovny pracující na sféře (např. **S2**, **spherely**).

### 7.2 Nevalidní geometrie

Operace GEOS předpokládají **validní** geometrie. Klasický problém je polygon tvaru „motýlek“, jehož hrana protíná sama sebe. Shapely ho bez varování vytvoří, ale výsledky výpočtů jsou nesmyslné:

In [ ]:
motylek = Polygon([(0, 0), (2, 2), (2, 0), (0, 2)])

print("Validní?   ", motylek.is_valid)
print("Proč ne?   ", shapely.is_valid_reason(motylek))
print("Plocha:    ", motylek.area, "  ← zjevně špatně, dva trojúhelníky mají plochu 2")

opraveny = shapely.make_valid(motylek)
print("Opraveno:  ", opraveny)
print("Plocha:    ", opraveny.area)

> V praxi se nevalidní geometrie objevují hlavně v datech z digitalizace nebo po konverzích. Před analýzou je dobré je zkontrolovat (`is_valid`) a případně opravit (`make_valid`, dříve trik `buffer(0)`).

### 7.3 Přesnost čísel s plovoucí desetinnou čárkou

Souřadnice jsou `float64`, takže bod, který „leží na čáře“, na ní kvůli zaokrouhlení ležet nemusí:

In [ ]:
usecka = LineString([(0, 0), (1, 3.3)])
bod_na_usecce = usecka.interpolate(0.37, normalized=True)    # bod ve 37 % délky úsečky

print("Bod:                    ", bod_na_usecce)
print("Úsečka obsahuje bod?    ", usecka.contains(bod_na_usecce))     # False!
print("Vzdálenost bodu od čáry:", usecka.distance(bod_na_usecce))
print("dwithin s tolerancí 1e-9:", shapely.dwithin(usecka, bod_na_usecce, 1e-9))

Řešením je pracovat s **tolerancí** (`dwithin`, `buffer` o malou hodnotu) nebo zaokrouhlit souřadnice na pevnou mřížku pomocí `shapely.set_precision`.

### 7.4 Pouze 2D

Shapely umí uložit souřadnici Z, ale **všechny výpočty probíhají ve 2D** – Z se ignoruje:

In [ ]:
svah = LineString([(0, 0, 0), (100, 0, 100)])   # stoupání 100 m na 100 m vodorovně
print("Má Z souřadnici?", svah.has_z)
print("Délka podle Shapely:", svah.length, "m  (skutečná 3D délka je", round(np.hypot(100, 100), 1), "m)")

### 7.5 Další omezení

* **Výkon cyklů v Pythonu** – jednotlivé objekty jsou pomalé; je potřeba používat vektorizované funkce a `STRtree` (viz 6.2 a 6.7). Pro opravdu velká data (stovky milionů prvků) se hodí databáze (PostGIS, DuckDB Spatial) nebo Apache Sedona.
* **Žádná topologie mezi prvky** – Shapely neví, že dva sousední kraje sdílejí hranici; zjednodušení každého zvlášť vytvoří mezery a překryvy (řeší např. knihovna `topojson`).
* **Žádné I/O, atributy, CRS ani rastry** – Shapely je „geometrický motor“, ne GIS. Kompletní práci zajišťuje až kombinace s GeoPandas, pyproj, rasteriem apod.
* **Pořadí výsledků** některých funkcí (např. `voronoi_polygons`) neodpovídá pořadí vstupů – je potřeba je spárovat zpětně (viz 6.6).

## 8. Alternativy a souvislosti

| Knihovna | Vztah ke Shapely |
|---|---|
| **GeoPandas** | staví na Shapely; přidává tabulky, CRS, čtení/zápis souborů a mapy |
| **PyGEOS** | dříve rychlá vektorizovaná alternativa, v roce 2022 sloučena do Shapely 2.0 |
| **pyproj** | převody souřadnic a geodetické výpočty na elipsoidu – doplňuje Shapely |
| **JTS** (Java) / **GEOS** (C++) | „motory“, na kterých Shapely stojí |
| **Turf.js** | podobná geometrická knihovna pro JavaScript |
| **PostGIS**, **DuckDB Spatial** | stejné operace (taktéž přes GEOS) přímo v databázi, vhodné pro velká data |
| **spherely / S2** | geometrie na sféře místo v rovině |

## 9. Shrnutí

* Shapely je **základní stavební kámen** prostorového Pythonu – rychlá rovinná geometrie nad knihovnou GEOS.
* Umí vytvářet geometrie, testovat vztahy (`contains`, `intersects`…) a provádět operace (`buffer`, `intersection`, `union_all`, `simplify`, `voronoi_polygons`…).
* Od verze 2.0 nabízí **vektorizované funkce a STRtree**, které jsou řádově rychlejší než cykly.
* **Limity:** nezná CRS (pozor na stupně!), počítá jen v rovině a ve 2D, vyžaduje validní geometrie a je citlivá na přesnost čísel. Neumí číst soubory ani pracovat s atributy – proto se téměř vždy používá spolu s GeoPandas a pyproj.

## Zdroje

* Dokumentace Shapely: https://shapely.readthedocs.io/
* Zdrojový kód: https://github.com/shapely/shapely
* GEOS: https://libgeos.org/
* Natural Earth (data): https://www.naturalearthdata.com/ , GitHub: https://github.com/nvkelso/natural-earth-vector
* pyproj: https://pyproj4.github.io/pyproj/
* GeoPandas: https://geopandas.org/